In [0]:
# Load from Databricks table
df = spark.table("r2_india")

display(df)


In [0]:
from pyspark.sql.functions import col

df_clean = df.select(
    col("YEAR"),
    col("JUN"),
    col("JUL"),
    col("AUG"),
    col("SEP"),
    col("ANNUAL")
)

display(df_clean)

In [0]:
from pyspark.sql.functions import col, when

df_numeric = df_clean.select(
    col("YEAR"),
    when(col("JUN") == "NA", None).otherwise(col("JUN")).cast("double").alias("JUN"),
    when(col("JUL") == "NA", None).otherwise(col("JUL")).cast("double").alias("JUL"),
    when(col("AUG") == "NA", None).otherwise(col("AUG")).cast("double").alias("AUG"),
    when(col("SEP") == "NA", None).otherwise(col("SEP")).cast("double").alias("SEP"),
    when(col("ANNUAL") == "NA", None).otherwise(col("ANNUAL")).cast("double").alias("ANNUAL")
)

display(df_numeric)

In [0]:
from pyspark.sql.functions import expr

df_features = df_numeric.withColumn(
    "monsoon_rainfall",
    expr("JUN + JUL + AUG + SEP")
)

display(df_features)

In [0]:
from pyspark.sql.functions import when

df_final = df_features.withColumn(
    "flood_risk",
    when(col("monsoon_rainfall") > 1500, 1).otherwise(0)
)

display(df_final)

In [0]:
# Remove rows with null values
df_ml_ready = df_final.dropna()

display(df_ml_ready)

In [0]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["monsoon_rainfall"],
    outputCol="features"
)

df_ml = assembler.transform(df_ml_ready)

display(df_ml)

In [0]:
train_data, test_data = df_ml.randomSplit([0.8, 0.2])
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="flood_risk")
model = lr.fit(train_data)

predictions = model.transform(test_data)

display(predictions.select("monsoon_rainfall", "flood_risk", "prediction"))

In [0]:
df_features.select("monsoon_rainfall").describe().show()

In [0]:
df_final.write.format("delta").mode("overwrite").saveAsTable("flood_prediction_data")

In [0]:
import matplotlib.pyplot as plt

# Convert to pandas (small data so fine)
pdf = df_final.select("monsoon_rainfall", "flood_risk").toPandas()

plt.figure(figsize=(8,5))
plt.scatter(pdf["monsoon_rainfall"], pdf["flood_risk"], alpha=0.5)
plt.xlabel("Monsoon Rainfall")
plt.ylabel("Flood Risk")
plt.title("Flood Risk vs Rainfall")
plt.show()

In [0]:
import matplotlib.pyplot as plt

pdf = df_final.select("monsoon_rainfall", "flood_risk").toPandas()

plt.figure(figsize=(8,5))

plt.hist(pdf[pdf["flood_risk"]==0]["monsoon_rainfall"], bins=30, alpha=0.6, label="No Flood")
plt.hist(pdf[pdf["flood_risk"]==1]["monsoon_rainfall"], bins=30, alpha=0.6, label="Flood")

plt.xlabel("Monsoon Rainfall")
plt.ylabel("Frequency")
plt.title("Flood vs Non-Flood Rainfall Distribution")
plt.legend()

plt.show()

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

pdf = df_final.select("monsoon_rainfall", "flood_risk").toPandas()

plt.figure(figsize=(6,5))
sns.boxplot(x="flood_risk", y="monsoon_rainfall", data=pdf)

plt.title("Rainfall Distribution by Flood Risk")
plt.xlabel("Flood Risk (0 = No, 1 = Yes)")
plt.ylabel("Monsoon Rainfall")

plt.show()